# Push NVD CVE data to Hugging Face

This notebook gathers every scraped NVD CVE CSV from `scrapers/scraped_data/CVE/` and publishes them as a single dataset on the Hugging Face Hub.

It mirrors `generate_dataset.ipynb`, which stages its data in a local `hf_upload/` folder. Here we stage the NVD data in a **new** local `nvd_hf_upload/` folder (hence the name) and upload that folder to the Hub.

### Requirements

`pandas` for reading/combining the CSVs, `huggingface_hub` for the upload, and `python-dotenv` to load the `HF_TOKEN` from the project `.env` file.

In [1]:
!pip install pandas
!pip install -q huggingface_hub python-dotenv

## Load every CVE file

All scraped NVD CSVs live under `scrapers/scraped_data/CVE/`. We discover them with a glob instead of hard-coding file names, so a new yearly file (for example `nvd_2027_cwe699.csv`) is picked up automatically the next time this notebook runs.

Every file shares the same columns:

- `cve_id`: CVE identifier.
- `published`: publication timestamp.
- `last_modified`: last modification timestamp.
- `published_year`: year the CVE was published.
- `description`: NVD vulnerability description.
- `all_nvd_cwes`: all CWE identifiers NVD associates with the CVE (pipe separated).
- `cwe_699_matches`: the subset of those CWEs inside the CWE-699 Software category (pipe separated).
- `number_of_nvd_cwes`: count of `all_nvd_cwes`.
- `number_of_cwe_699_matches`: count of `cwe_699_matches`.

We then concatenate all files into a single `CVE` DataFrame, resetting the index so row numbers stay unique across files.

In [1]:
import glob
from pathlib import Path
from utils import remove_capital_words, remove_parentheses
import pandas as pd

# == KAGGLE/COLAB ==
# !git clone https://github.com/regularpooria/CWE_CVE_Mapping
# BASE_DIR = "./CWE_CVE_Mapping"

# == LOCAL ==
BASE_DIR = "../"

CVE_DIR = Path(BASE_DIR) / "scrapers" / "scraped_data" / "CVE"
CVE_files = sorted(glob.glob(str(CVE_DIR / "*.csv")))

print(f"Found {len(CVE_files)} CVE files:")
for file in CVE_files:
    print(f"  - {Path(file).name}")

CVE: pd.DataFrame = pd.DataFrame()
for i, file in enumerate(CVE_files):
    if i == 0:
        CVE = pd.read_csv(file)
        continue

    CVE = pd.concat([CVE, pd.read_csv(file)], ignore_index=True)

print(f"\nCombined {len(CVE):,} rows x {len(CVE.columns)} columns")

Found 4 CVE files:
  - nvd_2023_cwe699.csv
  - nvd_2024_cwe699.csv
  - nvd_2025_cwe699.csv
  - nvd_2026_cwe699.csv

Combined 88,146 rows x 9 columns


### Inspect the combined data

A quick sanity check before uploading: preview the first rows and confirm the number of rows per year.

In [2]:
print(CVE["published_year"].value_counts().sort_index())
CVE.head(5)

published_year
2023    18282
2024    24906
2025    26816
2026    18142
Name: count, dtype: int64


,cve_id,published,last_modified,published_year,description,all_nvd_cwes,cwe_699_matches,number_of_nvd_cwes,number_of_cwe_699_matches
0,CVE-2023-0028,2023-01-01T01:15:12.627,2026-06-17T05:24:37.547,2023,Cross-site Scripting (XSS) - Stored in GitHub ...,CWE-79,CWE-79,1,1
1,CVE-2023-22451,2023-01-02T16:15:11.063,2026-06-17T05:35:28.713,2023,Kiwi TCMS is an open source test management sy...,CWE-521,CWE-521,1,1
2,CVE-2023-0038,2023-01-03T14:15:10.390,2026-06-17T05:24:38.627,2023,"The ""Survey Maker – Best WordPress Survey Plug...",CWE-79,CWE-79,1,1
3,CVE-2023-22456,2023-01-03T19:15:10.483,2026-06-17T05:35:29.547,2023,"ViewVC, a browser interface for CVS and Subver...",CWE-79,CWE-79,1,1
4,CVE-2023-0046,2023-01-04T12:15:08.613,2026-06-17T05:24:39.563,2023,Improper Restriction of Names for Files and Ot...,CWE-641,CWE-641,1,1


## Resolve CWE-699 children and categories

`CWE_SOFTWARE.json` stores the CWE-699 Software hierarchy. Each category has a `children` object whose keys are the child CWE ids that belong to it.

This step:

- Builds `child_to_cats`, mapping every child CWE id to the category (or categories) that contain it.
- Validates that no CVE has an empty `description`.
- Derives `true_children`: the `cwe_699_matches` of each CVE filtered down to ids that are known CWE-699 children.
- Validates that at least one known child remains for every CVE.
- Derives `true_categories`: the sorted set of category ids that those children belong to.

`cwe_699_matches` is pipe separated (for example `CWE-74|CWE-89`), so `parse_ids` extracts the numeric ids with a regex before they are checked against `child_to_cats`.

In [ ]:
import json
import re

with open(f"{BASE_DIR}/scrapers/scraped_data/CWE_SOFTWARE.json", "r", encoding="utf-8") as f:
    CWE = json.load(f)


def parse_ids(value):
    return re.findall(r"\d+", str(value))


child_to_cats = {}
for cat_id, category in CWE.items():
    for child_id in category["children"].keys():
        child_to_cats.setdefault(child_id, []).append(cat_id)

if CVE["description"].isna().any():
    raise ValueError("Some descriptions in CVE are empty, please check and remove")

CVE["true_children"] = CVE["cwe_699_matches"].apply(
    lambda v: [c for c in parse_ids(v) if c in child_to_cats]
)
if (CVE["true_children"].str.len() == 0).any():
    raise ValueError("Some CVEs have a cwe_699_matches value that isn't a known CWE-699 child")

CVE["true_categories"] = CVE["true_children"].apply(
    lambda cs: sorted({cat for c in cs for cat in child_to_cats[c]})
)

CVE["description_normalized"] = CVE["description"].apply(lambda x: remove_capital_words(remove_parentheses(x)))
CVE[["cve_id", "cwe_699_matches", "true_children", "true_categories"]].head(5)

## Write the dataset card

Hugging Face renders the `README.md` at the root of a dataset repository as its landing page. We write it into the staging folder so it is committed together with the CSV.

In [4]:
from pathlib import Path

# The new staging folder (the `hf_upload` equivalent for the NVD data).
UPLOAD_DIR = Path("nvd_hf_upload")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Dataset card shown on the Hub page.
DATASET_CARD = '''---
language:
- en
tags:
- cybersecurity
- vulnerability
- cve
- nvd
- cwe
- software-security
task_categories:
- text-classification
pretty_name: NVD CVE Dataset
license: apache-2.0
---

# NVD CVE Dataset

## Dataset description

This dataset contains CVE records scraped from the NVD (National Vulnerability Database) that map to weaknesses in the CWE-699 Software category. Each row is a single CVE with its description and its NVD and CWE-699 weakness mappings.

## Dataset structure

The dataset is provided as `NVD_CVE_Dataset.csv`. Its main fields include:

- `cve_id`: CVE identifier.
- `published`: publication timestamp.
- `last_modified`: last modification timestamp.
- `published_year`: year the CVE was published.
- `description`: NVD vulnerability description.
- `all_nvd_cwes`: all CWE identifiers NVD associates with the CVE (pipe separated).
- `cwe_699_matches`: the subset of those CWEs in the CWE-699 Software category (pipe separated).
- `number_of_nvd_cwes`: count of `all_nvd_cwes`.
- `number_of_cwe_699_matches`: count of `cwe_699_matches`.
- `true_children`: the CWE-699 child ids from `cwe_699_matches` that exist in `CWE_SOFTWARE.json` (list).
- `true_categories`: the CWE-699 category ids those children belong to (sorted list).

## Data sources

Raw CVE records are collected from the NVD. CWE-699 Software category membership is determined from the CWE data scraped by this project.

The dataset is generated by `dataset/push_nvd_to_huggingface.ipynb`.

## Intended uses

This dataset is intended for research and experimentation involving:

- CVE-to-CWE classification.
- Vulnerability search and retrieval.
- Security text embedding and similarity experiments.
- Analysis of software weakness categories.

## Limitations

The mappings reflect the upstream NVD data and may contain incomplete, ambiguous, or changing vulnerability information. The dataset should not be treated as a definitive security assessment or as a replacement for reviewing the original NVD and CWE records.

## License and attribution

Please review and follow the licenses and attribution requirements of the upstream NVD and CWE sources before redistributing or using this dataset.
'''

(UPLOAD_DIR / "README.md").write_text(DATASET_CARD, encoding="utf-8")
print(f"Wrote dataset card to {UPLOAD_DIR / 'README.md'}")

Wrote dataset card to nvd_hf_upload/README.md


## Save and upload the folder (`nvd_hf_upload`)

This is the NVD counterpart of the `hf_upload` cell in `generate_dataset.ipynb`. It saves the combined DataFrame as `NVD_CVE_Dataset.csv` inside the new `nvd_hf_upload/` folder and commits the whole folder to the Hugging Face dataset repository.

Configuration:

- `HF_TOKEN` is read from the project `.env` file (loaded with `python-dotenv`). Create it at the repository root with `HF_TOKEN=hf_...`.
- The target repository defaults to `regularpooria/NVD_CVE_699_Dataset`. Override it by adding `HF_NVD_DATASET_REPO_ID=your-user/your-repo` to `.env`.
- The upload creates the repository automatically if it does not exist yet (the token must have write access).

In [5]:
# Save the final NVD dataset locally and commit it to the Hugging Face dataset repository.
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import HfApi


load_dotenv()
HF_REPO_ID = "regularpooria/NVD_CVE_699_Dataset"


# New staging folder dedicated to the NVD data.
UPLOAD_DIR = Path("nvd_hf_upload")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = UPLOAD_DIR / "NVD_CVE_Dataset.csv"
CVE.to_csv(DATASET_PATH, index=False)


api = HfApi(token=os.environ.get("HF_TOKEN"))
commit_url = api.upload_folder(
    folder_path=str(UPLOAD_DIR),
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message="Update NVD CVE dataset",
)


print(f"Saved {len(CVE):,} rows to {DATASET_PATH}")
print(f"Committed dataset and dataset card to {commit_url}")

/home/regularpooria/Projects/CWE_CVE_Mapping/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 88,146 rows to nvd_hf_upload/NVD_CVE_Dataset.csv
Committed dataset and dataset card to https://huggingface.co/datasets/regularpooria/NVD_CVE_699_Dataset/commit/7452c05f49c1a25afa08a784131bd63cdf829abd
